## Setup

In [2]:
print("Hello World")

Hello World


In [3]:
import os
import re
import pandas as pd
import torch
from tqdm import tqdm
from transformers import (
	Qwen2_5_VLForConditionalGeneration, AutoProcessor, # For Qwen
	AutoModelForCausalLM, AutoTokenizer # For DeepSeek
)
from qwen_vl_utils import process_vision_info

In [4]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

IMAGE_DIR = "./math/images/images/"
train = pd.read_csv("./math/train.csv")
test = pd.read_csv("./math/test.csv")

# --- UPDATE THESE PATHS ---
QWEN_PATH = "/home/ub089/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5"
# Replace the hash below with your DeepSeek snapshot folder!
DEEPSEEK_PATH = "/home/ub089/.cache/huggingface/hub/models--deepseek-ai--deepseek-math-7b-instruct/snapshots/0a5828f800a36df0fd7f0ed581b983246c0677ff"

## Normalize

In [5]:
def normalize_answer(text):
	if pd.isna(text): return ""
	text = str(text).lower().strip()
	trans = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
	text = text.translate(trans)
	text = text.replace('$', '')
	for u in ['ตารางเซนติเมตร', 'ลูกบาศก์หน่วย', 'เซนติเมตร', 'องศา', 'หน่วย', 'จำนวน', 'วิธี', 'แบบ', 'ค่า', 'ร้อยละ', 'ดอลลาร์', 'บาท', 'degrees', 'square centimeters', 'years old']:
		text = text.replace(u, '')
	text = re.sub(r'\\frac{([^{}]+)}{([^{}]+)}', r'(\1)/(\2)', text) 
	text = text.replace(r'\sqrt', 'sqrt').replace(r'\pi', 'pi').replace(r'\times', '*').replace(r'\cdot', '*').replace(r'\div', '/').replace(r'\pm', '+-')
	text = text.replace(r'\overrightarrow', '').replace(r'\overline', '').replace(r'\vec', '')
	for token in [r'\left', r'\right', r'\,', r'\;', r'\:', r'\!']:
		text = text.replace(token, '')
	text = re.sub(r'[\s{}\\,]', '', text)
	text = re.sub(r'^\((\d+)\)$', r'\1', text)
	try:
		val = float(text)
		if val.is_integer(): text = str(int(val))
		else: text = str(val) 
	except ValueError: pass 
	return text

## Model

In [ ]:
print("Loading Qwen (The Extractor) onto GPU 0...")
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
	QWEN_PATH, torch_dtype=torch.bfloat16, device_map="cuda:0", 
	local_files_only=True
)
qwen_processor = AutoProcessor.from_pretrained(QWEN_PATH, local_files_only=True)
qwen_processor.tokenizer.padding_side = 'left'

print("Loading DeepSeek-Math (The Solver) onto GPU 1...")
ds_model = AutoModelForCausalLM.from_pretrained(
	DEEPSEEK_PATH, torch_dtype=torch.bfloat16, device_map="cuda:1", 
	local_files_only=True
)
ds_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK_PATH, local_files_only=True)
ds_tokenizer.padding_side = 'left'
# DeepSeek-Math doesn't have a default pad token, so we map it to eos
if ds_tokenizer.pad_token is None:
	ds_tokenizer.pad_token = ds_tokenizer.eos_token

Loading Qwen (The Extractor) onto GPU 0...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading DeepSeek-Math (The Solver) onto GPU 1...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Prompt

In [ ]:
qwen_prompt = (
	"Transcribe all the text, numbers, and mathematical equations in this image perfectly into LaTeX. "
	"If there is a geometric diagram, describe its properties, shapes, and measurements in text. "
	"Do NOT attempt to solve the problem. Only extract the information."
)

ds_system_prompt = (
	"You are an expert math solver. Solve the following math problem step-by-step. "
	"The problem is transcribed from an image and may contain Thai text and LaTeX. "
	"Write out your full reasoning. "
	"When you are finished, you MUST output the final answer on a new line starting exactly with 'FINAL_ANSWER: ' "
	"followed by the value.\n\n"
	"Problem Transcription:\n"
)

## Inference

In [10]:
BATCH_SIZE = 8

In [ ]:
print(f"Starting Two-Step Pipeline Validation...")

train['raw_prediction'] = "Error"
train['norm_prediction'] = "error"
train['norm_truth'] = train['answer'].apply(normalize_answer)
train['qwen_extraction'] = "" # Let's save what Qwen sees for debugging!

for i in tqdm(range(0, len(train), BATCH_SIZE)):
	batch_df = train.iloc[i : i + BATCH_SIZE]
	valid_indices = [] 
	qwen_messages = []

	for idx, row in batch_df.iterrows():
		img_path = f"{IMAGE_DIR}{row['id']}.jpg"
		if os.path.exists(img_path):
			valid_indices.append(idx)
			qwen_messages.append([{
				"role": "user",
				"content": [
					{"type": "image", "image": img_path},
					{"type": "text", "text": qwen_prompt}
				]
			}])

	if not qwen_messages: continue 
		
	q_texts = [qwen_processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in qwen_messages]
	image_inputs, video_inputs = process_vision_info(qwen_messages)

	q_inputs = qwen_processor(
		text=q_texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
	).to(qwen_model.device) # Goes to cuda:0

	with torch.no_grad():
		q_gen_ids = qwen_model.generate(**q_inputs, max_new_tokens=256, do_sample=False) 

	q_gen_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(q_inputs.input_ids, q_gen_ids)]
	extractions = qwen_processor.batch_decode(q_gen_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

	ds_prompts = [f"{ds_system_prompt}{ext}" for ext in extractions]
	
	ds_inputs = ds_tokenizer(ds_prompts, return_tensors="pt", padding=True).to(ds_model.device) # Goes to cuda:1

	with torch.no_grad():
		ds_gen_ids = ds_model.generate(**ds_inputs, max_new_tokens=1024, do_sample=False)

	ds_gen_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(ds_inputs.input_ids, ds_gen_ids)]
	final_outputs = ds_tokenizer.batch_decode(ds_gen_ids_trimmed, skip_special_tokens=True)


	for ext, out_text, df_idx in zip(extractions, final_outputs, valid_indices):
		if "FINAL_ANSWER:" in out_text:
			raw_ans = out_text.split("FINAL_ANSWER:")[-1].strip()
		else:
			raw_ans = out_text.split('\n')[-1].strip()

		train.at[df_idx, 'qwen_extraction'] = ext
		train.at[df_idx, 'raw_prediction'] = raw_ans
		train.at[df_idx, 'norm_prediction'] = normalize_answer(raw_ans)

Starting Two-Step Pipeline Validation...


  0%|          | 0/70 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
  3%|▎         | 2/70 [02:08<1:12:45, 64.20s/it]


KeyboardInterrupt: 

In [ ]:
train['is_correct'] = train['norm_prediction'] == train['norm_truth']
accuracy = train['is_correct'].mean()

print(f"\n======================================")
print(f"DUAL-PIPELINE VALIDATION ACCURACY: {accuracy * 100:.2f}%")
print(f"======================================\n")

# View a failure to see if Qwen extracted it wrong, or DeepSeek solved it wrong
failures = train[~train['is_correct']]
if not failures.empty:
	print("Debug - Reviewing the first failure:")
	fail_row = failures.iloc[0]
	print(f"ID: {fail_row['id']}")
	print(f"--- WHAT QWEN EXTRACTED (GPU 0) --- \n{fail_row['qwen_extraction']}")
	print(f"\n--- WHAT DEEPSEEK ANSWERED (GPU 1) --- \nRaw Prediction: {fail_row['raw_prediction']}")
	print(f"Norm Truth: {fail_row['norm_truth']} | Norm Pred: {fail_row['norm_prediction']}")

## Test

In [11]:
print(f"Starting Two-Step Inference on {len(test)} TEST images...")

# Pre-allocate array
predicted_answers = ["0"] * len(test)

for i in tqdm(range(0, len(test), BATCH_SIZE)):
	batch_df = test.iloc[i : i + BATCH_SIZE]

	batch_messages = []
	valid_indices = [] 

	# 1. STEP 1: QWEN EXTRACTION (GPU 0)
	for idx, row in batch_df.iterrows():
		img_path = f"{IMAGE_DIR}{row['id']}.jpg"
		if os.path.exists(img_path):
			valid_indices.append(idx)
			batch_messages.append([{
				"role": "user",
				"content": [
					{"type": "image", "image": img_path},
					{"type": "text", "text": qwen_prompt}
				]
			}])

	if not batch_messages: continue 

	q_texts = [qwen_processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
	image_inputs, video_inputs = process_vision_info(batch_messages)

	q_inputs = qwen_processor(
		text=q_texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
	).to(qwen_model.device)

	with torch.no_grad():
		q_gen_ids = qwen_model.generate(**q_inputs, max_new_tokens=256, do_sample=False) 

	q_gen_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(q_inputs.input_ids, q_gen_ids)]
	extractions = qwen_processor.batch_decode(q_gen_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

	# 2. STEP 2: DEEPSEEK SOLVING (GPU 1)
	ds_prompts = [f"{ds_system_prompt}{ext}" for ext in extractions]
	ds_inputs = ds_tokenizer(ds_prompts, return_tensors="pt", padding=True).to(ds_model.device)

	with torch.no_grad():
		ds_gen_ids = ds_model.generate(**ds_inputs, max_new_tokens=1024, do_sample=False)

	ds_gen_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(ds_inputs.input_ids, ds_gen_ids)]
	final_outputs = ds_tokenizer.batch_decode(ds_gen_ids_trimmed, skip_special_tokens=True)

	# 3. Parsing & Storing
	for out_text, df_idx in zip(final_outputs, valid_indices):
		if "FINAL_ANSWER:" in out_text:
			raw_ans = out_text.split("FINAL_ANSWER:")[-1].strip()
		else:
			raw_ans = out_text.split('\n')[-1].strip()

		clean_ans = normalize_answer(raw_ans)
		if clean_ans == "":
			clean_ans = "0"

		predicted_answers[df_idx] = clean_ans

Starting Two-Step Inference on 420 TEST images...


100%|██████████| 53/53 [54:47<00:00, 62.04s/it]


In [25]:
test['answer'] = predicted_answers
submission = test.copy().drop(columns=['image_path'])
submission.to_csv("submission.csv", index=False)

print("\nSuccessfully generated submission.csv! You can now download this file from your environment.")


Successfully generated submission.csv! You can now download this file from your environment.


In [26]:
def normalize_answer(raw: str) -> str:
	"""
	Convert DeepSeek verbose output to clean answer matching train.csv format.
	Handles: boxed extraction, LaTeX fractions/sqrt/pi, cut-off Thai fragments.
	"""
	if not raw or not isinstance(raw, str):
		return ""

	s = raw.strip()

	# ── Step 1: Extract boxed value ──────────────────────────────────────────
	extracted = None

	# \boxed{...} or boxed{...}
	m = re.findall(r'(?:\\)?boxed\{([^{}]+)\}', s)
	if m:
		extracted = m[-1].strip()

	if extracted is None:
		# "boxed(2)/(3)" or "boxed(0.7)" — boxed followed by opening paren
		m = re.search(r'boxed(\(.*)', s)
		if m:
			tail = m.group(1)
			frac = re.match(r'\(([^)]+)\)/\(([^)]+)\)', tail)
			if frac:
				extracted = f'({frac.group(1)})/({frac.group(2)})'
			else:
				single = re.match(r'\(([^)]+)\)', tail)
				if single:
					extracted = single.group(1).strip()

	if extracted is None:
		# "(boxed0.7)" — outer parens wrap the whole expression
		inner = re.match(r'^\((.+)\)$', s)
		if inner:
			m2 = re.search(r'boxed([^\u0E00-\u0E7F\s)]+)', inner.group(1))
			if m2:
				extracted = m2.group(1).strip().rstrip('.')

	if extracted is None:
		# bare: boxedXXX — stop before Thai chars, spaces, closing paren, dot
		m = re.search(r'boxed([^\u0E00-\u0E7F\s)]+)', s)
		if m:
			extracted = m.group(1).strip().rstrip('.')

	if extracted is not None:
		s = extracted

	# ── Step 2: Detect incomplete/cut-off Thai fragment ──────────────────────
	cut_endings = (
		'คือ', 'ได้', 'ว่า', 'และ', 'คื', 'เป', 'ของ',
		'แต่', 'จาก', 'ไป', 'มา', 'ใน', 'ที่'
	)
	thai_char_count = len(re.findall(r'[\u0E00-\u0E7F]', s))
	thai_heavy = thai_char_count > len(s) * 0.45

	if extracted is None:
		if thai_heavy and any(s.endswith(e) for e in cut_endings):
			return ""
		if thai_heavy and len(s) > 40:
			return ""

	# ── Step 3: (a)/(b) → \frac{a}{b} ───────────────────────────────────────
	s = re.sub(
		r'(-?)\(([^()]+)\)/\(([^()]+)\)',
		lambda m: m.group(1) + f'\\frac{{{m.group(2)}}}{{{m.group(3)}}}', s
	)

	# ── Step 4: sqrt notation → \sqrt{} ──────────────────────────────────────
	s = re.sub(r'sqrt\[(\d+)\]\{([^}]+)\}', lambda m: f'\\sqrt[{m.group(1)}]{{{m.group(2)}}}', s)
	s = re.sub(r'sqrt\(([^)]+)\)',           lambda m: f'\\sqrt{{{m.group(1)}}}', s)
	s = re.sub(r'sqrt(\d+)',                 lambda m: f'\\sqrt{{{m.group(1)}}}', s)
	s = re.sub(r'sqrt([a-z]\w*)',            lambda m: f'\\sqrt{{{m.group(1)}}}', s)

	# ── Step 5: pi → \pi ─────────────────────────────────────────────────────
	s = re.sub(r'(?<![a-zA-Z])pi(?![a-zA-Z])', r'\\pi', s)

	# ── Step 6: widehat → \widehat{} ─────────────────────────────────────────
	s = re.sub(r'widehat([a-zA-Z]+)', lambda m: f'\\widehat{{{m.group(1)}}}', s)

	# ── Step 7: Strip trailing English unit words after math ─────────────────
	s = re.sub(
		r'(?<=[0-9}\\])\s*(square\s*units?|cubic\s*units?|sq\.?\s*units?)\s*$',
		'', s, flags=re.IGNORECASE
	).strip()

	# ── Step 8: Strip leading boilerplate (fallback if no boxed found) ───────
	boilerplate = [
		r'thereforetheansweris', r'sotheansweris', r'thefinalansweris', r'theansweris',
		r'solution\s*:', r'ดังนั้นคำตอบคือ', r'ดังนั้น', r'คำตอบคือ',
		r'ฉะนั้นคำตอบคือ', r'ฉะนั้น', r'ตอบ',
	]
	for bp in boilerplate:
		s = re.sub(f'^{bp}', '', s, flags=re.IGNORECASE).strip()

	# ── Step 9: Misc cleanup ─────────────────────────────────────────────────
	s = re.sub(r'[\.。]+$', '', s).strip()   # trailing dots
	s = re.sub(r'^[\]\[]+', '', s).strip()   # leading stray brackets

	# ── Step 10: Wrap in $...$ if LaTeX notation detected ────────────────────
	has_latex = bool(re.search(
		r'\\frac|\\sqrt|\\pi|\\widehat|\^[\d{(]|_[\d{(]|\\cdot|\\times|\\pm', s
	))
	if has_latex and not s.startswith('$'):
		s = f'${s}$'

	# ── Step 11: Collapse whitespace ─────────────────────────────────────────
	s = re.sub(r'\s+', ' ', s).strip()
	return s


In [27]:
# Load your submission where 'answer' is raw DeepSeek output
submission = pd.read_csv("submission.csv")

# Apply normalizer to every answer
submission['answer'] = submission['answer'].apply(normalize_answer)

# Drop image_path if present, keep only id + answer
submission = submission[['id', 'answer']]
submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")

# Quick sanity check
print(submission.head(10).to_string())

submission.csv saved!
    id         answer
0    1  $\frac{2}{3}$
1   10              ค
2  100               
3  101           1620
4  103             84
5  109               
6  110  $100\sqrt{3}$
7  111            0.7
8  115              9
9  116          12a11


In [29]:
submission.sample(20)

,id,answer
345,618,3420
31,144,inf
72,211,
410,86,8
416,93,140
207,420,
184,392,1
329,595,15
252,479,10000000
165,362,1


In [37]:
submission = submission.fillna(2)

In [38]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      420 non-null    int64 
 1   answer  420 non-null    object
dtypes: int64(1), object(1)
memory usage: 6.7+ KB


In [39]:
submission.to_csv("submission.csv", index=False)

In [41]:
submission = pd.read_csv("submission.csv")
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      420 non-null    int64 
 1   answer  420 non-null    object
dtypes: int64(1), object(1)
memory usage: 6.7+ KB


In [42]:
submission.head(20)

,id,answer
0,1,$\frac{2}{3}$
1,10,ค
2,100,2
3,101,1620
4,103,84
5,109,2
6,110,$100\sqrt{3}$
7,111,0.7
8,115,9
9,116,12a11
